# 02 — Smoke Test

Overfit ~500 train samples on the `hemt_clip` variant to verify the full pipeline (HDF5 dataset → model → loss → backprop → optimiser) is wired correctly.

**Pass criterion:** train loss drops sharply and train accuracy hits >85% within a few epochs. Val accuracy will *not* be high — 500 samples isn't enough to generalise, the point is just to confirm the model can fit data.

If train loss won't budge, there's a wiring bug (frozen the wrong params, wrong loss, label dtype mismatch, etc.) — fix it here before burning compute units on the real run.

## Setup
One bootstrap cell handles everything: env vars, Drive mount (pick **Account B** in the OAuth popup), repo clone-or-pull, deps install, `jax`/`flax` removal (they force `numpy>=2` and break the pinned `numpy 1.26.4`), and copying the HDF5 from Drive to local SSD for faster random reads. Safe to re-run after any runtime recycle.

In [ ]:
# Bootstrap — idempotent. Safe to re-run on a fresh or warm Colab runtime.
# Handles: env vars, Drive mount, repo clone/pull, deps, jax/flax removal,
# and h5 copy from Drive to local SSD.
import os, sys, subprocess, shutil

# Env vars FIRST — must be set before any transformers import.
os.environ["USE_FLAX"] = "FALSE"
os.environ["USE_TF"] = "FALSE"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
REPO_URL = "https://github.com/staharizvi/hemt-clip-fnd.git"
REPO_DIR = "/content/hemt-clip-fnd"
H5_DRIVE = "/content/drive/MyDrive/hemt-clip-fnd/data/fakeddit.h5"
H5_LOCAL = "/content/fakeddit.h5"

if IN_COLAB:
    if not os.path.ismount("/content/drive"):
        from google.colab import drive
        drive.mount("/content/drive")
    else:
        print("Drive already mounted.")

    if os.path.exists(os.path.join(REPO_DIR, ".git")):
        print("Repo present — pulling latest…")
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--quiet"], check=True)
    else:
        print("Cloning repo…")
        subprocess.run(["git", "clone", "--quiet", REPO_URL, REPO_DIR], check=True)

    subprocess.run(["pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"], check=True)
    # jaxlib/jax/flax force numpy>=2 and clash with the pinned numpy 1.26.4
    subprocess.run(["pip", "uninstall", "-y", "-q", "jax", "jaxlib", "flax"], check=False)

    if not os.path.exists(H5_LOCAL):
        if os.path.exists(H5_DRIVE):
            print(f"Copying {H5_DRIVE} -> {H5_LOCAL}…")
            shutil.copy(H5_DRIVE, H5_LOCAL)
        else:
            print(f"WARNING: {H5_DRIVE} not found — run the data-prep notebook first.")
    else:
        print(f"h5 already at {H5_LOCAL}.")

    os.chdir(REPO_DIR)

print("\ncwd:", os.getcwd())
print("h5 :", H5_LOCAL, "exists:", os.path.exists(H5_LOCAL))

In [ ]:
import os, sys, random, time
import numpy as np
import torch
import torch.nn as nn
import yaml
from torch.utils.data import Subset, DataLoader

# Make the repo importable whether run from repo root or notebooks/
ROOT = os.getcwd()
if os.path.basename(ROOT) == 'notebooks':
    ROOT = os.path.dirname(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from data.dataset import HEMTClipDataset
from models.hemt_clip import build_from_config

print('cwd :', os.getcwd())
print('root:', ROOT)
print('cuda:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

In [ ]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

with open(os.path.join(ROOT, 'configs', 'base.yaml')) as f:
    cfg = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
HDF5_PATH = cfg['data']['hdf5_path']
print('hdf5:', HDF5_PATH)
print('exists:', os.path.exists(HDF5_PATH))

## Data — 500-sample train subset, 100-sample val

In [ ]:
train_full = HEMTClipDataset(
    HDF5_PATH, split='train',
    tokenizer_name=cfg['model']['text']['name'],
    max_text_len=cfg['data']['max_text_len'],
)
val_full = HEMTClipDataset(
    HDF5_PATH, split='val',
    tokenizer_name=cfg['model']['text']['name'],
    max_text_len=cfg['data']['max_text_len'],
)

N_TRAIN, N_VAL = 500, 100
rng = np.random.default_rng(SEED)
train_idx = rng.choice(len(train_full), size=N_TRAIN, replace=False).tolist()
val_idx   = rng.choice(len(val_full),   size=N_VAL,   replace=False).tolist()

train_ds = Subset(train_full, train_idx)
val_ds   = Subset(val_full,   val_idx)
print(f'train_ds={len(train_ds)}  val_ds={len(val_ds)}')

# Peek at one sample to confirm shapes / dtypes.
s = train_ds[0]
for k, v in s.items():
    print(f'  {k:15s} {tuple(v.shape) if v.ndim else v.shape}  {v.dtype}')

In [ ]:
BATCH = 16  # T4 handles this fine for the full model in fp16
# num_workers=0 keeps the notebook simple — 500 samples isn't I/O bound.
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=True)
print('batches: train=', len(train_loader), ' val=', len(val_loader))

## Model + optimiser + loss

In [ ]:
VARIANT = 'hemt_clip'
model = build_from_config(cfg, variant=VARIANT).to(device)
print(f'variant={VARIANT}  trainable={model.trainable_parameter_count()/1e6:.1f}M')

optim = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-4, weight_decay=0.01,
)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

## Training loop (overfit smoke test)

In [ ]:
EPOCHS = 5
GRAD_CLIP = 1.0

def move(batch, device):
    return {k: v.to(device, non_blocking=True) for k, v in batch.items()}

@torch.no_grad()
def evaluate(loader):
    model.eval()
    tot, corr, loss_sum = 0, 0, 0.0
    for batch in loader:
        batch = move(batch, device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type=='cuda'):
            out = model(batch)
            loss = criterion(out['logits'], batch['label'])
        loss_sum += loss.item() * batch['label'].size(0)
        corr += (out['logits'].argmax(-1) == batch['label']).sum().item()
        tot  += batch['label'].size(0)
    return loss_sum / tot, corr / tot

history = []
t0 = time.time()
for ep in range(1, EPOCHS + 1):
    model.train()
    tot, corr, loss_sum = 0, 0, 0.0
    for batch in train_loader:
        batch = move(batch, device)
        optim.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type=='cuda'):
            out = model(batch)
            loss = criterion(out['logits'], batch['label'])
        scaler.scale(loss).backward()
        scaler.unscale_(optim)
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], GRAD_CLIP,
        )
        scaler.step(optim)
        scaler.update()
        loss_sum += loss.item() * batch['label'].size(0)
        corr += (out['logits'].argmax(-1) == batch['label']).sum().item()
        tot  += batch['label'].size(0)
    tr_loss, tr_acc = loss_sum / tot, corr / tot
    va_loss, va_acc = evaluate(val_loader)
    history.append((ep, tr_loss, tr_acc, va_loss, va_acc))
    print(f'epoch {ep:>2}  '
          f'train_loss={tr_loss:.4f}  train_acc={tr_acc:.3f}  |  '
          f'val_loss={va_loss:.4f}  val_acc={va_acc:.3f}  '
          f'({time.time()-t0:.0f}s)')

final_tr_acc = history[-1][2]
if final_tr_acc < 0.85:
    print(f'\nSMOKE TEST FAILED: train_acc={final_tr_acc:.3f} after {EPOCHS} epochs (expected >0.85).')
    print('Likely causes: frozen the wrong params, label dtype mismatch, alpha all-zero, lr too low.')
else:
    print(f'\nSMOKE TEST PASSED: train_acc={final_tr_acc:.3f}. Pipeline is sound — safe to scale up.')

## What to look for

**Good run (T4):**
- Epoch 1 train_loss ~0.7, train_acc ~55–65% (random-ish start).
- Epoch 5 train_loss < 0.25, train_acc > 90%.
- Each epoch ~30–60s.
- val_acc bouncing around 50% — expected on 100 random val samples with a model overfit on 500.

**Bad signals:**
- Train loss flat across epochs → optimiser not stepping, or all params frozen.
- Train loss explodes (NaN/Inf) → fp16 overflow; drop to fp32 by setting `enabled=False` on autocast/scaler.
- Train acc stays at 50% but loss drops → model is collapsing to one class; check label balance in the subset.

Once this passes, the next step is `training/train.py`: same loop pattern, plus two-stage fine-tuning, TensorBoard, per-epoch Drive checkpoints, and resume-from-checkpoint.